# Naive RAG walk (S2.2)

Same six stages as `labs/02_naive_pipeline.py`. After Stage 4, open `store/naive/manifest.json`.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "ragbench").is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

from ragbench.chunkers import chunk_corpus
from ragbench.corpus import load_documents
from ragbench.embed import ToyEmbedder
from ragbench.generate import generate_answer
from ragbench.settings import LAB_EMBEDDER
from ragbench.store import print_card, save_index

QUESTION = "What was ACME revenue growth in Q2 2023?"

In [ ]:
docs = load_documents()
print(len(docs), "documents from data/acme/")
for doc in docs:
    print(f"{doc.doc_id:18}  {doc.path}")

In [ ]:
chunks = chunk_corpus(docs, "fixed", size=80, overlap=0)
print("chunker=fixed size=80 overlap=0 ->", len(chunks), "chunks")
for c in chunks:
    if "3%" in c.text:
        print(c.chunk_id, "acme=", "acme" in c.text.lower(), "q2=", "q2" in c.text.lower())
        print(c.text[:220])

In [ ]:
print(LAB_EMBEDDER["name"], "dim", LAB_EMBEDDER["dim"], "semantic_mode", LAB_EMBEDDER["semantic_mode"])
print(LAB_EMBEDDER["why"])
print("swap:", ", ".join(LAB_EMBEDDER["production_swap"]))
embedder = ToyEmbedder(semantic_mode=True)
vectors = embedder.encode([c.text for c in chunks])
print("vectors.shape", tuple(vectors.shape))

In [ ]:
index = save_index(
    "naive",
    chunks,
    vectors,
    extra={
        "chunker": "fixed",
        "chunk_kwargs": {"size": 80, "overlap": 0},
        "contextual": False,
        "search": "dense",
        "doc_ids": [d.doc_id for d in docs],
        "doc_count": len(docs),
    },
)
print(print_card(index))

In [ ]:
hits = index.dense_search(QUESTION, k=3)
for i, hit in enumerate(hits, start=1):
    print(i, round(hit.score, 3), hit.chunk.chunk_id)
    print(" ", hit.chunk.text.replace("\n", " ")[:160])
answer, meta = generate_answer(QUESTION, [h.chunk for h in hits], mode="extractive")
print(meta)
print(answer)